In [ ]:
from game import Connect4
import numpy as np
import math
import time

import torch
print(torch.__version__)

import torch.nn as nn
import torch.nn.functional as F

2.13.0


In [ ]:
from turtle import forward


class ResNet(nn.module):
    def __init__(self, game, num_resBlocks, num_hidden):
        super().__init__()
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1), # 3 is the number of input planes that feed in
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for _ in range(num_resBlocks)]
        )

        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BachNorm2d(32),
            nn.ReLU(),
            nn.Flatten(), # why?
            nn.Linear(32 * game.row_count * game.column_count, game.action_size),
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.row_count * game.column_count,  1),
            nn.Tanh()
        )

    
    def forward(self, x):
        x = self.startBlock(x)
        
        for resBlock in self.backBone:
            x = resBlock(x)

        policy = self.policyHead(x)
        value = self.valueHead(x)

        return policy, value

        
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))

        x += residual
        x = F.relu(x) #why F.relu and nn.relu?



    
     

In [ ]:
def sample_move(valid_moves):
    chosen_move = np.random.choice(np.flatnonzero(valid_moves))
    #print(f"playing move {chosen_move}")
    return chosen_move

class Node:
    def __init__(self, args, game, state, player, action_taken=None, parent = None) -> None:

        self.game = game
        self.args = args
        self.state = state.copy()
        self.player = player

        self.visit_count = 0
        self.win_count = 0
        self.parent = parent
        self.children = []
        self.untried_actions = self.game.get_valid_moves(self.state) # this looks like: [True, True, False, ...]
        self.action_taken = action_taken


    def visit(self):
        self.visit_count += 1

    def is_terminal(self):
        if not self.action_taken:
            return False
        
        _, terminated = self.game.get_value_and_terminated(self.state, self.action_taken)
        return terminated

    def is_fully_expanded(self):
        return np.sum(self.untried_actions) == 0

    def has_children(self):
        return len(self.children) > 0

    def best_child(self):
        if np.sum(self.untried_actions) != 0:
            raise "need to explore all children first"
        
        if len(self.children) == 0:
            raise "no children to explore!"

        for child in self.children:
            if child.visit_count == 0:
                return child

        def ucb(child):
            exploit = child.win_count / child.visit_count
            explore = self.args['c'] * math.sqrt(math.log(self.visit_count) / child.visit_count)
            return exploit + explore

        return max(self.children, key=ucb)
    
    def expand(self):
        if self.is_fully_expanded():
            raise

        sampled_action = sample_move(self.untried_actions)
        #print(f"untried actions durign expansion: {self.untried_actions}")
        self.untried_actions[sampled_action] = False
        state = self.game.make_move(self.state, self.player, sampled_action)
        
        child_node = Node(
            args = self.args, 
            game = self.game,
            state = state,
            player = self.game.get_opponent(self.player),
            action_taken=sampled_action,
            parent=self
        )

        self.children.append(child_node)

        return child_node

    
    def rollout(self):
        
        state = self.state
        action_taken = self.action_taken
        player = self.player

        while True:
            value, terminated = self.game.get_value_and_terminated(state, action_taken)
            if terminated:
                #print(f"game finished, returning {value * player * -1}")
                return value * player * -1

            valid_moves = self.game.get_valid_moves(state)
            action_taken = sample_move(valid_moves)

            state = self.game.make_move(state, player, action_taken)
            player = self.game.get_opponent(player)

            #self.game.print_board(state)

        


    def UCB(self):
        if self.visit_count == 0:
            return np.inf
        pass

    def backpropogate(self, result):
        self.win_count += result
        self.visit()

        if self.parent is not None:
            self.parent.backpropogate(-1 * result)
        

In [ ]:
class MCTS:
    def __init__(self, game, args) -> None:
        self.game = game
        self.args = args

    def search(self, state, verbose = False):
       
        # define root
        root = Node(self.args, self.game, state, player=1)

        for i in range(self.args['iterations']):
            if verbose:
                #time.sleep(1)
                print(f"iteration {i} started")
            current_node = root
            
            while not current_node.is_terminal() and current_node.is_fully_expanded():
                current_node = current_node.best_child()
                if verbose:
                    self.game.print_board(current_node.state)
            
            if not current_node.is_terminal():
                current_node = current_node.expand()
            
            result = current_node.rollout()
            
            current_node.backpropogate(result)
            if verbose:
                print(f"iteration {i} finished, result = {result}")
        

        visits = np.zeros(self.game.col_count)
        for child in root.children:
            visits[child.action_taken] = child.visit_count

        return visits


In [ ]:
game = Connect4()
state_init = game.get_initial_state()

round_1 = MCTS(game=game, args={'iterations' : 2000, 'c' : 1.41})
results = round_1.search(state=state_init, verbose=False)

In [ ]:
results

In [ ]:
connect4 = Connect4()
state = connect4.get_initial_state()
mcts = MCTS(game=connect4, args={'iterations' : 8000, 'c' : 1.41})


player = 1 
while True:
    if player == -1:
        legal_moves = connect4.get_valid_moves(state)
        print(f"Legal Moves: {[i for i in range(connect4.col_count) if legal_moves[i]]}")
        
        action = int(input(f"player {player}: "))
        if action < 0 or action >= connect4.col_count or not legal_moves[action]:
            print('illegal move')
            continue

    else:
        search = mcts.search(state=state)
        action = np.argmax(search)

    state = connect4.make_move(state, player, action)
    print(f"player {player} plays {action}")
    connect4.print_board(state)

    value, terminated = connect4.get_value_and_terminated(state, action)

    if terminated:
        if value == 1:
            print(f"Player {player} wins!")
        else:
            print("draw")
        break


    player = connect4.get_opponent(player)
